# Capítulo 9 · Avaliação de testes diagnósticos e rastreamento: caderno complementar
*Ciência de Dados e Inteligência Artificial em Saúde, Volume 1*

> **Este caderno acompanha o capítulo do livro; não o reproduz.** Definições, deduções e explicações estão no livro. Aqui você executa os cálculos do caso e confere os números. Todos os dados são **sintéticos** e os valores têm apenas finalidade didática.

A **Parte A** percorre o caso do capítulo, a avaliação de um teste rápido e de um biomarcador de ferida para infecção de sítio cirúrgico, pelas perguntas da diretoria de qualidade; aparecem só as seções com cálculo (9.3 a 9.7). O livro desenha as figuras; o caderno imprime os mesmos números. A **Parte B** traz a resolução escrita de todos os exercícios do capítulo (1 a 12; os enunciados estão no livro) e, nos exercícios cuja resolução precisa de cálculo (3, 4, 7, 8, 9, 11 e 12), o código e o resultado. O código é Python 3 com pandas, statsmodels, scikit-learn e scipy. Execute as células em ordem.

## Parte A. O caso

**Dados.** `data/accuracy_cohort.csv`: 1.200 visitas consecutivas de seguimento em 30 dias, com o biomarcador da ferida (`marker_ng_ml`), o teste rápido (`rapid_test`), a infecção verdadeira (`ssi`, o padrão de referência) e outros campos. `data/two_gate.csv`: 100 infecções profundas e 100 voluntários saudáveis. `data/screening_cohort.csv`: 10.000 mulheres acompanhadas por 20 anos, com os momentos do início pré-clínico, do diagnóstico clínico e da morte. Cada célula traz o número da seção do livro a que pertence.

### 9.3 A tabela dois por dois de um teste: sensibilidade e especificidade

*Quão bem o teste rápido detecta a infecção e a descarta?*

In [1]:
# 1. Ler a coorte de acurácia: 1.200 visitas de seguimento, com o teste rápido (teste índice) e a infecção verdadeira (padrão de referência)
import pandas as pd
from statsmodels.stats.proportion import proportion_confint

d = pd.read_csv("data/accuracy_cohort.csv")
inf = d["ssi"] == "yes"
pos = d["rapid_test"] == "positive"

# 2. As quatro células da tabela dois por dois: uma coluna booleana somada é uma contagem
tp = (pos & inf).sum()
fn = (~pos & inf).sum()
fp = (pos & ~inf).sum()
tn = (~pos & ~inf).sum()
print("Verdadeiros positivos:", tp, "; falsos positivos:", fp, "; falsos negativos:", fn, "; verdadeiros negativos:", tn)

# 3. Sensibilidade, especificidade e os valores preditivos
se = tp / (tp + fn)
sp = tn / (tn + fp)
ppv = tp / (tp + fp)
npv = tn / (tn + fn)
print("Sensibilidade:", round(se, 3), "; especificidade:", round(sp, 3), "; valor preditivo positivo:", round(ppv, 3), "; valor preditivo negativo:", round(npv, 3))

# 4. Intervalos de Wilson de 95% das quatro proporções
for name, k, n in [("Sensibilidade", tp, tp + fn), ("Especificidade", tn, tn + fp), ("Valor preditivo positivo", tp, tp + fp), ("Valor preditivo negativo", tn, tn + fn)]:
    lo, hi = proportion_confint(k, n, alpha=0.05, method="wilson")
    print(name + ":", round(k / n, 3), "(IC de 95%", round(lo, 3), "a", round(hi, 3), ")")

Verdadeiros positivos: 67 ; falsos positivos: 193 ; falsos negativos: 27 ; verdadeiros negativos: 913
Sensibilidade: 0.713 ; especificidade: 0.825 ; valor preditivo positivo: 0.258 ; valor preditivo negativo: 0.971
Sensibilidade: 0.713 (IC de 95% 0.614 a 0.794 )
Especificidade: 0.825 (IC de 95% 0.802 a 0.847 )
Valor preditivo positivo: 0.258 (IC de 95% 0.208 a 0.314 )
Valor preditivo negativo: 0.971 (IC de 95% 0.959 a 0.98 )


### 9.4 Valores preditivos e prevalência

*O que significa um resultado positivo ou negativo em diferentes prevalências?*

In [2]:
# 1. Valores preditivos do teste rápido em diferentes prevalências, a partir de sua sensibilidade e especificidade
for prev in [0.01, 0.03, 0.078, 0.20, 0.40]:
    ppv_prev = prev * se / (prev * se + (1 - prev) * (1 - sp))
    npv_prev = (1 - prev) * sp / ((1 - prev) * sp + prev * (1 - se))
    print("Prevalência", prev, ": valor preditivo positivo", round(ppv_prev, 3), "; valor preditivo negativo", round(npv_prev, 3))

Prevalência 0.01 : valor preditivo positivo 0.04 ; valor preditivo negativo 0.996
Prevalência 0.03 : valor preditivo positivo 0.112 ; valor preditivo negativo 0.989
Prevalência 0.078 : valor preditivo positivo 0.257 ; valor preditivo negativo 0.971
Prevalência 0.2 : valor preditivo positivo 0.505 ; valor preditivo negativo 0.92
Prevalência 0.4 : valor preditivo positivo 0.731 ; valor preditivo negativo 0.812


### 9.5 Razões de verossimilhança

*Quanto cada resultado muda as chances de infecção?*

In [3]:
# 1. As razões de verossimilhança do teste binário
lr_pos = se / (1 - sp)
lr_neg = (1 - se) / sp
print("Razão de verossimilhança positiva:", round(lr_pos, 2), "; razão de verossimilhança negativa:", round(lr_neg, 2))

# 2. A razão de verossimilhança de cada faixa do biomarcador (ng/mL), a partir das contagens de pacientes infectados e não infectados na faixa
bands = [(0, 20), (20, 40), (40, 80), (80, float("inf"))]
n_inf = inf.sum()
n_not = (~inf).sum()
for lo, hi in bands:
    in_band = (d["marker_ng_ml"] >= lo) & (d["marker_ng_ml"] < hi)
    a = (in_band & inf).sum()
    b = (in_band & ~inf).sum()
    print("Faixa", lo, "a", hi, ": infectados", a, "; não infectados", b, "; proporções", round(a / n_inf, 3), round(b / n_not, 3), "; razão de verossimilhança", round((a / n_inf) / (b / n_not), 2))

Razão de verossimilhança positiva: 4.08 ; razão de verossimilhança negativa: 0.35
Faixa 0 a 20 : infectados 10 ; não infectados 663 ; proporções 0.106 0.599 ; razão de verossimilhança 0.18
Faixa 20 a 40 : infectados 27 ; não infectados 362 ; proporções 0.287 0.327 ; razão de verossimilhança 0.88
Faixa 40 a 80 : infectados 37 ; não infectados 75 ; proporções 0.394 0.068 ; razão de verossimilhança 5.8
Faixa 80 a inf : infectados 20 ; não infectados 6 ; proporções 0.213 0.005 ; razão de verossimilhança 39.22


### 9.6 Da probabilidade pré-teste à pós-teste: frequências naturais e chances

*Qual é a probabilidade de infecção depois do teste, partindo da probabilidade antes dele?*

In [4]:
# 1. Frequências naturais: 1.000 pacientes com probabilidade pré-teste de 7,8%
n = 1000
infected = round(n * 0.078)
print("Infectados:", infected, "; verdadeiros positivos:", round(infected * se), "; falsos positivos:", round((n - infected) * (1 - sp)))

# 2. Probabilidade pós-teste a partir da probabilidade pré-teste e de uma razão de verossimilhança, na forma de chances
cases = [("7,8%, fita positiva", 0.078, lr_pos), ("7,8%, fita negativa", 0.078, lr_neg),
         ("7,8%, faixa superior", 0.078, 39.22), ("7,8%, faixa inferior", 0.078, 0.18),
         ("30%, fita positiva", 0.30, lr_pos), ("30%, fita negativa", 0.30, lr_neg)]
for label, prev, lr in cases:
    pre_odds = prev / (1 - prev)
    post_odds = pre_odds * lr
    print("Pré-teste", label + ": probabilidade pós-teste", round(post_odds / (1 + post_odds), 3))

Infectados: 78 ; verdadeiros positivos: 56 ; falsos positivos: 161
Pré-teste 7,8%, fita positiva: probabilidade pós-teste 0.257
Pré-teste 7,8%, fita negativa: probabilidade pós-teste 0.029
Pré-teste 7,8%, faixa superior: probabilidade pós-teste 0.768
Pré-teste 7,8%, faixa inferior: probabilidade pós-teste 0.015
Pré-teste 30%, fita positiva: probabilidade pós-teste 0.636
Pré-teste 30%, fita negativa: probabilidade pós-teste 0.13


In [5]:
# 3. Uma simulação de 12 meses de internações rastreadas para MRSA (semente 8): portadores, verdadeiros positivos e falsos positivos em cada mês
import numpy as np

rng = np.random.default_rng(8)
adm = 2000
prev_mrsa = 0.03
se_m = 0.90
sp_m = 0.97
carriers = rng.binomial(adm, prev_mrsa, size=12)
true_pos = rng.binomial(carriers, se_m)
false_pos = rng.binomial(adm - carriers, 1 - sp_m)
positives = true_pos + false_pos
print("Médias mensais - portadores:", round(carriers.mean(), 1), "; verdadeiros positivos:", round(true_pos.mean(), 1), "; falsos positivos:", round(false_pos.mean(), 1), "; positivos:", round(positives.mean(), 1))
print("Os positivos variam de", positives.min(), "a", positives.max(), "; parcela dos positivos que são portadores verdadeiros:", round((true_pos / positives).mean(), 3))

# 4. Leitos-dia de isolamento: 7 dias para um portador verdadeiro, 2 dias para um falso positivo (até o resultado da cultura)
iso_days = true_pos * 7 + false_pos * 2
print("Leitos-dia de isolamento por mês - média:", round(iso_days.mean(), 1), "; máximo:", iso_days.max(), "=", round(iso_days.max() / 30, 1), "leitos em um mês de 30 dias")

Médias mensais - portadores: 60.5 ; verdadeiros positivos: 55.2 ; falsos positivos: 59.9 ; positivos: 115.2
Os positivos variam de 97 a 129 ; parcela dos positivos que são portadores verdadeiros: 0.48
Leitos-dia de isolamento por mês - média: 506.6 ; máximo: 568 = 18.9 leitos em um mês de 30 dias


### 9.7 Escolhendo um ponto de corte: a curva ROC e a área sob ela

*Como um biomarcador contínuo se comporta ao longo dos pontos de corte?*

In [6]:
# 1. Sensibilidade e especificidade do biomarcador em quatro pontos de corte
from sklearn.metrics import roc_curve, roc_auc_score
from scipy import stats

x = d["marker_ng_ml"].to_numpy()
y = inf.to_numpy()
for c in [15, 30, 45, 60]:
    print("Ponto de corte", c, ": sensibilidade", round((x[y] >= c).mean(), 3), "; especificidade", round((x[~y] < c).mean(), 3))

# 2. A curva ROC, a área sob ela e a contagem de Mann-Whitney dividida por n1 vezes n0
fpr, tpr, thr = roc_curve(y, x)
auc = roc_auc_score(y, x)
u = stats.mannwhitneyu(x[y], x[~y], alternative="two-sided").statistic
n1 = y.sum()
n0 = (~y).sum()
print("Limiares retidos:", len(thr), "; área sob a curva:", round(auc, 3), "; contagem de Mann-Whitney sobre n1 vezes n0:", round(u / (n1 * n0), 3))

# 3. O erro padrão de Hanley-McNeil e o intervalo de 95% da área
q1 = auc / (2 - auc)
q2 = 2 * auc ** 2 / (1 + auc)
se_auc = np.sqrt((auc * (1 - auc) + (n1 - 1) * (q1 - auc ** 2) + (n0 - 1) * (q2 - auc ** 2)) / (n1 * n0))
print("Erro padrão da área:", round(se_auc, 4), "; intervalo de 95%:", round(auc - 1.96 * se_auc, 3), "a", round(auc + 1.96 * se_auc, 3))

Ponto de corte 15 : sensibilidade 1.0 ; especificidade 0.413
Ponto de corte 30 : sensibilidade 0.713 ; especificidade 0.825
Ponto de corte 45 : sensibilidade 0.564 ; especificidade 0.95
Ponto de corte 60 : sensibilidade 0.372 ; especificidade 0.984
Limiares retidos: 315 ; área sob a curva: 0.876 ; contagem de Mann-Whitney sobre n1 vezes n0: 0.876
Erro padrão da área: 0.0235 ; intervalo de 95%: 0.83 a 0.922


## Parte B. Soluções dos exercícios

### 9.15 Exercício 1. Um ensaio de troponina na chegada

Condição-alvo: infarto agudo do miocárdio em pacientes que chegam com dor torácica,
segundo uma definição clínica declarada. Teste índice: a troponina
à beira do leito, lida no ponto de corte fixo na chegada. Padrão de referência: o
diagnóstico final por um painel de adjudicação que usa a série laboratorial de troponina
de alta sensibilidade, eletrocardiografia, imagem e
acompanhamento de eventos em 30 dias, aplicado a todo paciente, independentemente do
resultado da chegada. População: pacientes consecutivos que chegam ao
pronto-socorro com dor torácica e nos quais se considera a alta, já que é onde o teste
será usado. Descartar o
infarto do miocárdio, por si só, não estabelece que a alta é segura:
outras síndromes coronarianas agudas e outras causas graves de dor torácica
exigem avaliação clínica. Uma
imperfeição: a referência depende de uma troponina seriada cujo próprio
ponto de corte e cuja temporização classificam erroneamente alguns pacientes de apresentação precoce, e um painel
que vê o resultado da chegada tem um problema de incorporação. Um estudo
em pacientes internados na unidade coronariana contém apenas a extremidade grave
do espectro, sem apresentações de baixo risco; sua sensibilidade
seria inflada pelo efeito espectro e sua especificidade seria
estimada nos pacientes errados, de modo que ele não pode dizer quantos
pacientes com alta seriam deixados passar.

### 9.15 Exercício 2. A tabela da fita urinária

Referência positiva 240: fita positiva 204 (verdadeiros positivos),
negativa 36 (falsos negativos). Referência negativa 400: fita
positiva 88 (falsos positivos), negativa 312 (verdadeiros negativos).
Sensibilidade $204/240=0{,}850$ e proporção de falsos negativos $36/240=0{,}150$,
com denominador as 240 mulheres com cultura positiva. Especificidade $312/400=0{,}780$
e proporção de falsos positivos $88/400=0{,}220$, com denominador as 400
mulheres com cultura negativa. Valor preditivo positivo $204/(204+88)=204/292
=0{,}699$, com denominador as 292 mulheres com fita positiva; valor preditivo
negativo $312/(312+36)=312/348=0{,}897$, com denominador as 348
mulheres com fita negativa.

### 9.15 Exercício 3. Intervalos de Wilson para a fita urinária

Sensibilidade 0,85 (IC de 95% 0,80-0,89), especificidade 0,78 (IC de 95%
0,74-0,82). O intervalo da especificidade é mais estreito porque se apoia em
400 mulheres contra 240, e porque 0,78 está mais próximo de 0,5 do que 0,85,
o que age no sentido contrário, mas com menos força. Um estudo de 640 mulheres
ainda deixa a sensibilidade conhecida apenas com margem de cerca de cinco pontos,
porque somente as mulheres com cultura positiva contribuem para ela; o tamanho
total não é o número relevante.

In [7]:
# 1. Intervalos de Wilson de 95% para a fita: 204 de 240 e 312 de 400
for k, n in [(204, 240), (312, 400)]:
    lo, hi = proportion_confint(k, n, alpha=0.05, method="wilson")
    print(k, "de", n, ":", round(k / n, 3), "(IC de 95%", round(lo, 3), "a", round(hi, 3), ")")

204 de 240 : 0.85 (IC de 95% 0.799 a 0.89 )
312 de 400 : 0.78 (IC de 95% 0.737 a 0.818 )


### 9.15 Exercício 4. A fita urinária em dois cenários

À mão, na instituição de longa permanência com $p=0{,}55$:
$\mathrm{VPP}=0{,}55\times0{,}85/(0{,}55\times0{,}85+0{,}45\times0{,}22)
=0{,}4675/(0{,}4675+0{,}099)=0{,}825$ e
$\mathrm{VPN}=0{,}45\times0{,}78/(0{,}45\times0{,}78+0{,}55\times0{,}15)
=0{,}351/(0{,}351+0{,}0825)=0{,}810$. Na triagem telefônica com $p=0{,}10$:
$\mathrm{VPP}=0{,}085/(0{,}085+0{,}198)=0{,}300$ e
$\mathrm{VPN}=0{,}702/(0{,}702+0{,}015)=0{,}979$.

Na instituição de longa permanência, uma fita positiva significa infecção em cerca de cinco
residentes em seis, e uma negativa ainda deixa uma chance de um em cinco;
na triagem, uma fita positiva significa infecção em menos de um entre três
que ligam, enquanto uma negativa é tranquilizadora. A mesma fita é um
auxílio para confirmar em um cenário e para excluir no outro, desde que
a sensibilidade e a especificidade sejam transportáveis para as duas populações.

In [8]:
# 1. A fita (sensibilidade 0,85, especificidade 0,78) em dois cenários com prevalência diferente
for prev in [0.55, 0.10]:
    ppv_prev = prev * 0.85 / (prev * 0.85 + (1 - prev) * (1 - 0.78))
    npv_prev = (1 - prev) * 0.78 / ((1 - prev) * 0.78 + prev * (1 - 0.85))
    print("Prevalência", prev, ": valor preditivo positivo", round(ppv_prev, 3), "; valor preditivo negativo", round(npv_prev, 3))

Prevalência 0.55 : valor preditivo positivo 0.825 ; valor preditivo negativo 0.81
Prevalência 0.1 : valor preditivo positivo 0.3 ; valor preditivo negativo 0.979


### 9.15 Exercício 5. Faixas de D-dímero

Proporções entre os 300 com embolia e os 1.200 sem: abaixo de 500,
$12/300=0{,}040$ e $720/1{.}200=0{,}600$, razão 0,07; 500 a 999,
$78/300=0{,}260$ e $360/1{.}200=0{,}300$, razão 0,87; 1.000 ou mais,
$210/300=0{,}700$ e $120/1{.}200=0{,}100$, razão 7,0. Dicotomizando em 500:
sensibilidade $288/300=0{,}960$, especificidade $720/1{.}200=0{,}600$,
$\mathrm{RV}^{+}=0{,}960/0{,}400=2{,}4$ e $\mathrm{RV}^{-}=0{,}040/0{,}600
=0{,}067$. A versão binária preserva a forte exclusão de um resultado baixo,
mas funde uma faixa intermediária quase não informativa (0,87) com uma faixa superior
fortemente positiva (7,0) em uma única razão positiva fraca de 2,4.

### 9.15 Exercício 6. Dois pacientes com 15\%

Árvore de 1.000 pacientes: 150 com embolia, 850 sem. Um resultado de
1.200 ng/mL cai na faixa superior, obtida por $150\times0{,}700=105$
pacientes com embolia e $850\times0{,}100=85$ sem: probabilidade
pós-teste $105/190=0{,}553$. Um resultado de 300 cai na faixa mais baixa,
obtida por $150\times0{,}040=6$ com embolia e $850\times0{,}600=510$
sem: $6/516=0{,}0116$. Pelas chances: chance pré-teste $0{,}15/0{,}85=0{,}176$;
multiplicada por 7,0 resulta em 1,235, probabilidade $1{,}235/2{,}235=0{,}553$; multiplicada por 0,067
resulta em 0,0118, probabilidade 0,0116. Os dois caminhos concordam, como
a Proposição 9.2 exige.

### 9.15 Exercício 7. Rastreamento de admissões para um organismo resistente

À mão: $1{.}500\times0{,}02=30$ portadores, dos quais $30\times0{,}92=27{,}6$
têm teste positivo; $1{.}470\times0{,}05=73{,}5$ falsos positivos; VPP
$27{,}6/101{,}1=0{,}273$.

Cerca de 28 verdadeiros e 76 falsos positivos por mês, 290 dias-leito de isolamento
em média e até 359 no pior mês, com 52% dos
dias-leito gerados por falsos positivos: com prevalência de 2%, uma especificidade
de 0,95 faz o programa gastar mais com o isolamento de não portadores do que
de portadores.

In [9]:
# 1. Rastrear 1.500 internações por mês para um organismo resistente (semente 8): portadores, verdadeiros positivos, falsos positivos
rng = np.random.default_rng(8)
adm = 1500
prev_c = 0.02
se_c = 0.92
sp_c = 0.95
carriers = rng.binomial(adm, prev_c, size=12)
true_pos = rng.binomial(carriers, se_c)
false_pos = rng.binomial(adm - carriers, 1 - sp_c)

# 2. Leitos-dia de isolamento: 5 dias para um portador verdadeiro, 2 dias para um falso positivo
iso_days = true_pos * 5 + false_pos * 2
print("Monthly means - true positives:", round(true_pos.mean(), 1), "; falsos positivos:", round(false_pos.mean(), 1), "; leitos-dia de isolamento:", round(iso_days.mean(), 1), "; pior mês:", iso_days.max())
print("Parte dos leitos-dia gerada por falsos positivos:", round((false_pos * 2).sum() / iso_days.sum(), 3))

Monthly means - true positives: 27.7 ; falsos positivos: 75.6 ; leitos-dia de isolamento: 289.5 ; pior mês: 359
Parte dos leitos-dia gerada por falsos positivos: 0.522


### 9.15 Exercício 8. O biomarcador segundo o diabetes

À mão, em 25 ng/mL: 75 dos 94 pacientes infectados estão nesse valor ou acima,
sensibilidade 0,798, e 812 dos 1.106 não infectados estão abaixo, especificidade
0,734; em 40: 57 de 94, sensibilidade 0,606, e 1.025 de 1.106,
especificidade 0,927.

A área é 0,861 (IC de 95% 0,789-0,933) nos pacientes com diabetes e
0,882 (IC de 95% 0,821-0,943) nos sem; a contagem de Mann-Whitney dividida por
$n_1n_0$ reproduz cada uma. A estimativa pontual é ligeiramente menor nos
pacientes com diabetes, mas cada grupo contém apenas cerca de 50 infecções.
A sobreposição dos intervalos separados não é um teste formal da diferença
entre eles e não estabelece discriminação igual. O código
salva as duas curvas dos subgrupos em `figures/exercise08_roc.pdf`.

In [10]:
# 1. Sensibilidade e especificidade do biomarcador em pontos de corte de 25 e 40 ng/mL (contagens e proporções)
for c in [25, 40]:
    print("Ponto de corte", c, ": sensibilidade", (x[inf] >= c).sum(), round((x[inf] >= c).mean(), 3), "; especificidade", (x[~inf] < c).sum(), round((x[~inf] < c).mean(), 3))

# 2. A área sob a curva ROC em pacientes com e sem diabetes, com a conferência de Mann-Whitney e o intervalo de 95%
for g in ["yes", "no"]:
    m = (d["diabetes"] == g).to_numpy()
    yy = inf[m]
    xx = x[m]
    auc_g = roc_auc_score(yy, xx)
    n1_g = yy.sum()
    n0_g = (~yy).sum()
    u_g = stats.mannwhitneyu(xx[yy], xx[~yy]).statistic
    q1_g = auc_g / (2 - auc_g)
    q2_g = 2 * auc_g ** 2 / (1 + auc_g)
    se_g = np.sqrt((auc_g * (1 - auc_g) + (n1_g - 1) * (q1_g - auc_g ** 2) + (n0_g - 1) * (q2_g - auc_g ** 2)) / (n1_g * n0_g))
    print("Diabetes", g + ": infectados", n1_g, "; não infectados", n0_g, "; área", round(auc_g, 3), "; Mann-Whitney", round(u_g / (n1_g * n0_g), 3), "; intervalo de 95%", round(auc_g - 1.96 * se_g, 3), "a", round(auc_g + 1.96 * se_g, 3))

Ponto de corte 25 : sensibilidade 75 0.798 ; especificidade 812 0.734
Ponto de corte 40 : sensibilidade 57 0.606 ; especificidade 1025 0.927
Diabetes yes: infectados 43 ; não infectados 267 ; área 0.861 ; Mann-Whitney 0.861 ; intervalo de 95% 0.789 a 0.933
Diabetes no: infectados 51 ; não infectados 839 ; área 0.882 ; Mann-Whitney 0.882 ; intervalo de 95% 0.821 a 0.943


### 9.15 Exercício 9. Uma razão de custos de cinco

Com uma infecção não detectada custando cinco vezes mais que um alarme falso, com prevalência de 7,8%,
$w=2{,}35$ e o ponto de corte que minimiza o custo é 44,9 ng/mL,
sensibilidade 0,574 e especificidade 0,950, mais alto que o de Youden, 32,3,
porque a chance contra a infecção (11,8) ainda excede a razão de custos
(5), de modo que os alarmes falsos dominam o custo total. Entre feridas encaminhadas, com
30%, $w=0{,}70/(0{,}30\times5)=0{,}467$, e o ponto de corte cai para 15,4, onde
a sensibilidade é 1,0 e a especificidade 0,432: com a infecção comum e
custosa, quase toda ferida deveria ser revisada, e o biomarcador acrescenta
pouco a uma política de revisar todas.

In [11]:
# 1. O ponto de corte que maximiza a sensibilidade ponderada quando um falso positivo custa um quinto de um caso perdido
fpr, tpr, thr = roc_curve(inf, x)
prev = inf.mean()
w = (1 - prev) / (prev * 5)
k = np.argmax(tpr - w * fpr)
j = np.argmax(tpr - fpr)
print("Peso:", round(w, 2), "; ponto de corte:", round(thr[k], 1), "; sensitivity:", round(tpr[k], 3), "; especificidade:", round(1 - fpr[k], 3))

# 2. O ponto de corte de Youden (custos iguais)
print("Ponto de corte de Youden:", round(thr[j], 1), "; sensitivity:", round(tpr[j], 3), "; especificidade:", round(1 - fpr[j], 3))

# 3. O mesmo com prevalência de 30%
w2 = 0.70 / (0.30 * 5)
k2 = np.argmax(tpr - w2 * fpr)
print("Peso:", round(w2, 3), "; ponto de corte:", round(thr[k2], 1), "; sensitivity:", round(tpr[k2], 3), "; especificidade:", round(1 - fpr[k2], 3))

Peso: 2.35 ; ponto de corte: 44.9 ; sensitivity: 0.574 ; especificidade: 0.95
Ponto de corte de Youden: 32.3 ; sensitivity: 0.713 ; especificidade: 0.855
Peso: 0.467 ; ponto de corte: 15.4 ; sensitivity: 1.0 ; especificidade: 0.432


### 9.15 Exercício 10. Três avaliações de testes para câncer

(a) Delineamento caso-controle de dois portões com espectro extremo: casos
metastáticos e doadores saudáveis. O efeito espectro infla tanto a sensibilidade
(casos fáceis) quanto a especificidade (não casos fáceis); domínio 1 do QUADAS-2,
seleção de pacientes (“evitou-se um delineamento caso-controle?” e
aplicabilidade). (b) Delineamento consecutivo de portão único com verificação
diferencial: positivos biopsiados, negativos acompanhados por telefone. O
acompanhamento telefônico deixa passar cânceres entre os negativos do teste, de modo que os falsos
negativos são subcontados e a sensibilidade é inflada; a especificidade é
inflada também sob esse mecanismo: cânceres não detectados entre os negativos
do teste são reclassificados como verdadeiros negativos, aumentando o numerador
e o denominador da especificidade, enquanto os falsos positivos permanecem inalterados.
O domínio 3 do QUADAS-2 trata da acurácia da referência telefônica;
o domínio 4, fluxo e temporização, pergunta se todos os pacientes receberam o mesmo
padrão de referência. Outros erros da referência poderiam mudar a direção. (c) Delineamento consecutivo
de portão único com verificação completa por biópsia, mas sem mascaramento do
patologista: o conhecimento do resultado do exame de sangue pode puxar a referência
para a concordância, inflando tanto a sensibilidade quanto a especificidade por
viés de revisão. Domínio 3 do QUADAS-2, padrão de referência (“os
resultados do padrão de referência foram interpretados sem conhecimento dos resultados
do teste índice?”).

### 9.15 Exercício 11. Tamanho de amostra para um teste rápido de influenza

A sensibilidade exige 137 pacientes com influenza, o que, com prevalência de 12%,
significa incluir 1.142 pacientes febris; a especificidade exige
203 sem influenza, que 231 inclusões fornecem. O estudo deve
incluir o maior número, 1.142. Em uma temporada branda, com 6%, a exigência
de influenza permanece 137, mas a inclusão dobra para 2.284,
porque metade dos incluídos contribui para a sensibilidade.
Esses totais produzem os tamanhos de grupo necessários em expectativa, não com
certeza; o recrutamento efetivo dos dois grupos deve ser monitorado.

In [12]:
# 1. Tamanho da amostra para um teste rápido de influenza: pacientes com influenza para a sensibilidade e o recrutamento com prevalência de 12%
z = 1.96
n_inf = z ** 2 * 0.85 * 0.15 / 0.06 ** 2
print("Pacientes com influenza:", round(n_inf, 1), "->", int(np.ceil(n_inf)), "; pacientes febris a recrutar com prevalência de 12%:", int(np.ceil(np.ceil(n_inf) / 0.12)))

# 2. Pacientes sem influenza para a especificidade e o recrutamento
n_not = z ** 2 * 0.95 * 0.05 / 0.03 ** 2
print("Pacientes sem influenza:", round(n_not, 1), "->", int(np.ceil(n_not)), "; pacientes febris a recrutar:", int(np.ceil(np.ceil(n_not) / 0.88)))

# 3. Em uma temporada leve, com prevalência de 6%
print("Pacientes febris a recrutar com prevalência de 6%:", int(np.ceil(np.ceil(n_inf) / 0.06)))

Pacientes com influenza: 136.1 -> 137 ; pacientes febris a recrutar com prevalência de 12%: 1142
Pacientes sem influenza: 202.8 -> 203 ; pacientes febris a recrutar: 231
Pacientes febris a recrutar com prevalência de 6%: 2284


### 9.15 Exercício 12. Quatro perguntas curtas

(a) Estratégia seriada sob independência: sensibilidade
$0{,}90\times0{,}86=0{,}774$; especificidade $1-(1-0{,}50)(1-0{,}95)=1-0{,}025
=0{,}975$. A sensibilidade observada pode diferir porque as crianças que o
escore deixa passar (poucos sintomas) são também aquelas com baixa carga bacteriana
que o teste de antígeno deixa passar, de modo que o segundo teste exclui menos das
crianças infectadas com escore positivo do que a independência prevê, e a
sensibilidade observada provavelmente é maior que 0,774.

(b) Rastreamento anual na coorte sintética.

O rastreamento anual detecta 820 tumores (69%) em vez de 644, com um tempo
de antecipação médio de 2,96 anos. A detecção mais precoce do mesmo tumor
alongaria seu tempo de antecipação. O valor médio ligeiramente menor aqui diz respeito
a um grupo detectado diferente, que inclui mais tumores rápidos, e a resultados
aleatórios de rastreamento diferentes; não é evidência de detecção mais tardia com
rastreamento mais frequente.
A sobrevida em cinco anos é 65,9% entre os detectados no rastreamento e 37,8% entre os
casos detectados clinicamente; as 688 mortes em 20 anos permanecem
inalteradas, porque o programa desloca diagnósticos e nada mais.

(c) Número necessário para rastrear $10{.}000/25=400$ pessoas durante 10 anos por
morte evitada; $60/25=2{,}4$ casos sobrediagnosticados por morte evitada.
Os números, por si sós, não estabelecem se o teste é aceitável para
a população nem se existem instalações adequadas de diagnóstico e tratamento.
Ambos são critérios de Wilson-Jungner.

(d) Limiar de tratamento $0{,}04/(0{,}04+0{,}08)=0{,}333$. Limiar de teste:
a probabilidade pré-teste cuja chance multiplicada por 6 alcança a chance de
$0{,}333/0{,}667=0{,}5$, isto é, chance $0{,}0833$ e probabilidade $0{,}077$.
Limiar de teste-tratamento: a probabilidade pré-teste cuja chance multiplicada por
0,2 ainda é igual a 0,5, chance 2,5 e probabilidade $0{,}714$. Vale a pena executar o teste
para probabilidades pré-teste entre cerca de 8% e 71% sob o
pressuposto declarado de ausência de custo, dano ou atraso do teste; abaixo de 8%, mesmo
um resultado positivo deixa a probabilidade abaixo do limiar de
tratamento, e acima de 71%, mesmo um resultado negativo a deixa acima.

In [13]:
# 1. Rastreamento anual na coorte sintética de 10.000 mulheres acompanhadas por 20 anos (semente 8): a cada ano, 85% dos tumores em sua fase detectável são encontrados
import pandas as pd

s = pd.read_csv("data/screening_cohort.csv")
c = s[s["cancer"] == "yes"].copy()
rng = np.random.default_rng(8)
onset = c["t_onset"].to_numpy()
t_clin = c["t_clinical"].to_numpy()
t_death = c["t_death"].to_numpy()
t_screen = np.full(len(c), np.nan)
for sc in np.arange(1.0, 20.1, 1.0):
    in_phase = (onset <= sc) & (sc < t_clin) & np.isnan(t_screen)
    hit = in_phase & (rng.random(len(c)) < 0.85)
    t_screen[hit] = sc

# 2. Tumores detectados, sua parte e o tempo de antecipação médio (anos)
det = ~np.isnan(t_screen)
t_dx = np.where(det, t_screen, t_clin)
print("Tumores detectados no rastreamento:", det.sum(), "; parte:", round(det.mean(), 3), "; tempo de antecipação médio:", round((t_clin - t_screen)[det].mean(), 2), "anos")

# 3. Sobrevida além de 5 anos do diagnóstico entre os detectados no rastreamento e os detectados clinicamente, e óbitos em 20 anos
print("Sobrevida além de 5 anos - detectados no rastreamento:", round(((t_death - t_dx) > 5)[det].mean(), 3), "; detectados clinicamente:", round(((t_death - t_dx) > 5)[~det].mean(), 3), "; óbitos em 20 anos:", int((t_death <= 20).sum()))

# 4. O limiar de tratamento a partir do dano de 0,04 de tratar um paciente sem a condição e do benefício de 0,08, e o limiar de teste
treat_thr = 0.04 / (0.04 + 0.08)
print("Limiar de tratamento:", round(treat_thr, 3), "; limiar de teste com razão de verossimilhança 6:", round(treat_thr / (treat_thr + (1 - treat_thr) * 6), 3), "; com 0,2:", round(treat_thr / (treat_thr + (1 - treat_thr) * 0.2), 3))

Tumores detectados no rastreamento: 820 ; parte: 0.69 ; tempo de antecipação médio: 2.96 anos
Sobrevida além de 5 anos - detectados no rastreamento: 0.659 ; detectados clinicamente: 0.378 ; óbitos em 20 anos: 688
Limiar de tratamento: 0.333 ; limiar de teste com razão de verossimilhança 6: 0.077 ; com 0,2: 0.714


---
*Dados sintéticos (`generate_data.py`, semente 8). Valores apenas didáticos; nada aqui é estimativa para um serviço real.*